### ETL: bronze.weather_history -> silver.weather_history_cleaned

In [0]:
import xml.etree.ElementTree as ET
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, FloatType, DateType
from pyspark.sql.functions import col, regexp_extract, explode, from_xml
import sys
import os
from pathlib import Path
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
# XML schema with array Meteoros as String
xml_schema_simplified = StructType([
    StructField("dia", ArrayType(
        StructType([
            StructField("_Dia", StringType()),
            StructField("hora", ArrayType(
                StructType([
                    StructField("_Hora", StringType()),
                    StructField("Meteoros", StringType()) 
                ])
            ))
        ])
    ))
])

In [0]:
df_wh_source = spark.table("dbw_routemind_euskadi_dev.bronze.weather_history")

In [0]:
df_parsed = df_wh_source.withColumn("xml_data", from_xml(col("raw_xml"), xml_schema_simplified))

In [0]:
extract_day = df_parsed.select(col("sensor_id"), explode(col("xml_data.dia")).alias("dia_data"))
extract_hour = extract_day.select(
    col("sensor_id"),
    col("dia_data._Dia").alias("date"),
    explode(col("dia_data.hora")).alias("hora_data")
)

In [0]:
df_wh_extracted = extract_hour.select(
    col("sensor_id").alias("sensorId"),
    col("date").cast("date"),
    col("hora_data._Hora").alias("time"),
    regexp_extract(col("hora_data.Meteoros"), r"<Tem\.Aire\._a_\d+cm>([^<]+)</Tem\.Aire\.", 1).cast("float").alias("temperature"),
    regexp_extract(col("hora_data.Meteoros"), r"<Precip\.\._a_\d+cm>([^<]+)</Precip\.", 1).cast("float").alias("precipitation")
).filter(
    col("temperature").isNotNull() | col("precipitation").isNotNull()
)

In [0]:
notnull_columns = [
    "sensorId",
    "date",
    "time"
]

df_clean = drop_null_required(df_wh_extracted, notnull_columns)

In [0]:
df_clean.count()

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.weather_history_cleaned"
delta_path = "abfss://silver@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/wheather_history/data"

df_clean.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(target_table)


print(f"APPEND completed on {target_table}. rows processed: {df_clean.count()}")